# Pretrained Embedding + XGBoost

Self-contained notebook: extract frozen AST/CLAP/ViT embeddings, then train XGBoost.

In [1]:
# Dataset root - edit this first if your Kaggle input path changes.
DATASET_ROOT = "/kaggle/input/datasets/kurt54/visual-audio/img_audio"
OUTPUT_DIR = "/kaggle/working/outputs/variants"
CLASSIFIER_NAME = "xgboost"
SEED = 42
BATCH_SIZE = 4
NUM_WORKERS = 2
MAX_SAMPLES = 0
CACHE_VERSION = "v2_mel_gate_traincrop"
EMBEDDING_KINDS = ("audio", "video", "fusion")

import json
import math
import random
import wave
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

LABELS = ("ambient", "leaf", "trunk", "twig")
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
PAPER_AVERAGE = "weighted"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

@dataclass
class DataConfig:
    data_root: Path
    target_sample_rate: int = 16000
    audio_window_sec: float = 0.8
    n_mels: int = 128
    n_fft: int = 1024
    hop_length: int = 256
    image_size: int = 224
    train_crop: str = "random"
    eval_crop: str = "energy"
    skip_missing_files: bool = True
    spectral_gate: bool = True
    spectral_gate_noise_percentile: float = 20.0
    spectral_gate_strength: float = 1.0

def find_data_root(data_root):
    data_root = Path(data_root)
    candidates = [data_root, data_root / "raw_dataset", data_root / "prepared_data", data_root / "dataset"]
    candidates += [Path("/kaggle/input/datasets/anhduy54/visual-audio/raw_dataset"), Path("/kaggle/input/visual-audio/raw_dataset")]
    for candidate in candidates:
        if (candidate / "audio_visual_dataset_default" / "dataset.csv").exists():
            return candidate
    return data_root

def build_index(data_root, skip_missing_files=True):
    rows = []
    for split_name in ("audio_visual_dataset_default", "audio_visual_dataset_robo_default"):
        split_dir = Path(data_root) / split_name
        csv_path = split_dir / "dataset.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        for item in df.to_dict("records"):
            audio_path = split_dir / item["audio_file"]
            image_path = split_dir / item["image_file"]
            if skip_missing_files and not (audio_path.exists() and image_path.exists()):
                continue
            rows.append({"split_name": split_name, "audio_path": str(audio_path), "image_path": str(image_path), "label": item["category"], "label_id": LABEL_TO_ID[item["category"]]})
    return pd.DataFrame(rows)

def resample_waveform(waveform, src_rate, dst_rate):
    if src_rate == dst_rate:
        return waveform.float()
    try:
        import torchaudio
        return torchaudio.transforms.Resample(orig_freq=src_rate, new_freq=dst_rate, lowpass_filter_width=64, rolloff=0.9475937167399596, resampling_method="sinc_interp_kaiser")(waveform.float())
    except Exception:
        from scipy.signal import resample_poly
        gcd = math.gcd(src_rate, dst_rate)
        y = resample_poly(waveform.squeeze(0).numpy(), dst_rate // gcd, src_rate // gcd).astype("float32")
        return torch.from_numpy(y).unsqueeze(0)

def read_wave(path):
    try:
        import torchaudio
        waveform, sample_rate = torchaudio.load(path)
        return waveform.mean(dim=0, keepdim=True), int(sample_rate)
    except Exception:
        with wave.open(str(path), "rb") as handle:
            sample_rate = handle.getframerate()
            channels = handle.getnchannels()
            width = handle.getsampwidth()
            frames = handle.readframes(handle.getnframes())
        dtype = np.int16 if width == 2 else np.uint8
        audio = np.frombuffer(frames, dtype=dtype).astype("float32")
        if channels > 1:
            audio = audio.reshape(-1, channels).mean(axis=1)
        if width == 2:
            audio = audio / 32768.0
        else:
            audio = (audio - 128.0) / 128.0
        return torch.from_numpy(audio).unsqueeze(0), sample_rate

def apply_spectral_gate(waveform, cfg):
    if (not cfg.spectral_gate) or waveform.shape[-1] < cfg.n_fft:
        return waveform
    window = torch.hann_window(cfg.n_fft, device=waveform.device)
    spec = torch.stft(waveform, n_fft=cfg.n_fft, hop_length=cfg.hop_length, win_length=cfg.n_fft, window=window, return_complex=True)
    magnitude = spec.abs()
    noise = torch.quantile(magnitude, cfg.spectral_gate_noise_percentile / 100.0, dim=-1, keepdim=True)
    gated_mag = (magnitude - cfg.spectral_gate_strength * noise).clamp_min(0.0)
    phase = spec / magnitude.clamp_min(1e-8)
    return torch.istft(gated_mag * phase, n_fft=cfg.n_fft, hop_length=cfg.hop_length, win_length=cfg.n_fft, window=window, length=waveform.shape[-1])

def crop_waveform(waveform, window_samples, mode):
    length = waveform.shape[-1]
    if length < window_samples:
        return torch.nn.functional.pad(waveform, (0, window_samples - length))
    if length == window_samples:
        return waveform
    if mode == "random":
        start = random.randint(0, length - window_samples)
    elif mode == "energy":
        energy = waveform.pow(2).mean(dim=0, keepdim=True).unsqueeze(0)
        kernel = torch.ones(1, 1, window_samples, device=waveform.device)
        start = int(torch.nn.functional.conv1d(energy, kernel).argmax(dim=-1).item())
    else:
        start = (length - window_samples) // 2
    return waveform[..., start:start + window_samples]

def waveform_to_mel(waveform, cfg):
    import torchaudio
    mel = torchaudio.transforms.MelSpectrogram(sample_rate=cfg.target_sample_rate, n_fft=cfg.n_fft, hop_length=cfg.hop_length, n_mels=cfg.n_mels, power=2.0)(waveform)
    mel = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)(mel)
    mel = (mel + 80.0) / 80.0
    return mel.clamp(0.0, 1.0)

class AudioVisualDataset(Dataset):
    def __init__(self, frame, cfg, crop_mode):
        self.frame = frame.reset_index(drop=True)
        self.cfg = cfg
        self.crop_mode = crop_mode
        self.image_transform = transforms.Compose([
            transforms.Resize((cfg.image_size, cfg.image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        waveform, sample_rate = read_wave(row.audio_path)
        waveform = resample_waveform(waveform, sample_rate, self.cfg.target_sample_rate)
        waveform = apply_spectral_gate(waveform, self.cfg)
        waveform = crop_waveform(waveform, int(round(self.cfg.target_sample_rate * self.cfg.audio_window_sec)), self.crop_mode)
        image = self.image_transform(Image.open(row.image_path).convert("RGB"))
        mel = waveform_to_mel(waveform, self.cfg)
        return {"waveform": waveform.squeeze(0), "audio": mel, "image": image, "label": torch.tensor(row.label_id, dtype=torch.long)}

def extract_model_embedding(output):
    if torch.is_tensor(output):
        return output
    pooler = getattr(output, "pooler_output", None)
    if torch.is_tensor(pooler):
        return pooler
    last_hidden = getattr(output, "last_hidden_state", None)
    if torch.is_tensor(last_hidden):
        return last_hidden[:, 0]
    if isinstance(output, (tuple, list)):
        for item in output:
            if torch.is_tensor(item):
                return item[:, 0] if item.ndim == 3 else item
    raise TypeError(f"Could not extract embedding from {type(output)!r}")

class PretrainedEmbeddingExtractor(nn.Module):
    def __init__(self, sample_rate=16000):
        super().__init__()
        from transformers import ASTFeatureExtractor, ASTModel, AutoProcessor, ClapModel
        self.sample_rate = sample_rate
        self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        self.ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        self.clap_processor = AutoProcessor.from_pretrained("laion/clap-htsat-unfused")
        self.clap_model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
        self.ast_sample_rate = getattr(self.ast_feature_extractor, "sampling_rate", sample_rate)
        self.clap_sample_rate = getattr(getattr(self.clap_processor, "feature_extractor", None), "sampling_rate", sample_rate)
        weights = models.ViT_B_16_Weights.DEFAULT
        self.image_model = models.vit_b_16(weights=weights)
        self.image_model.heads = nn.Identity()
        for module in (self.ast_model, self.clap_model, self.image_model):
            module.eval()
            for param in module.parameters():
                param.requires_grad = False
    def processor_arrays(self, waveform, target_rate):
        arrays = [item.detach().float().cpu().numpy() for item in waveform]
        if target_rate == self.sample_rate:
            return arrays
        from scipy.signal import resample_poly
        gcd = math.gcd(self.sample_rate, target_rate)
        return [resample_poly(item, target_rate // gcd, self.sample_rate // gcd).astype("float32") for item in arrays]
    def encode_ast(self, waveform):
        arrays = self.processor_arrays(waveform, self.ast_sample_rate)
        inputs = self.ast_feature_extractor(arrays, sampling_rate=self.ast_sample_rate, return_tensors="pt", padding=True)
        inputs = {key: value.to(waveform.device) for key, value in inputs.items()}
        return extract_model_embedding(self.ast_model(**inputs))
    def encode_ast_mel(self, mel):
        if mel.ndim == 3:
            mel = mel.unsqueeze(1)
        mel_db = mel.squeeze(1)
        if mel_db.min() >= 0.0 and mel_db.max() <= 1.0:
            mel_db = mel_db * 80.0 - 80.0
        input_values = mel_db.transpose(1, 2)
        max_length = int(getattr(self.ast_feature_extractor, "max_length", input_values.shape[1]))
        if input_values.shape[1] > max_length:
            input_values = input_values[:, :max_length, :]
        elif input_values.shape[1] < max_length:
            pad = input_values.new_zeros(input_values.shape[0], max_length - input_values.shape[1], input_values.shape[2])
            input_values = torch.cat([input_values, pad], dim=1)
        mean = float(getattr(self.ast_feature_extractor, "mean", 0.0))
        std = float(getattr(self.ast_feature_extractor, "std", 1.0))
        input_values = (input_values - mean) / max(std, 1e-8)
        return extract_model_embedding(self.ast_model(input_values=input_values.to(mel.device)))
    def encode_clap(self, waveform):
        arrays = self.processor_arrays(waveform, self.clap_sample_rate)
        inputs = self.clap_processor(audio=arrays, sampling_rate=self.clap_sample_rate, return_tensors="pt", padding=True)
        inputs = {key: value.to(waveform.device) for key, value in inputs.items()}
        return extract_model_embedding(self.clap_model.get_audio_features(**inputs))
    def forward(self, waveform, audio, image, kind):
        parts = []
        if kind in ("audio", "fusion"):
            parts.extend([self.encode_ast_mel(audio), self.encode_clap(waveform)])
        if kind in ("video", "fusion"):
            parts.append(self.image_model(image))
        return torch.cat(parts, dim=1)

def sample_stratified_frame(frame, max_samples, seed):
    if not max_samples or len(frame) <= max_samples:
        return frame.reset_index(drop=True)
    return frame.groupby("label", group_keys=False).apply(lambda x: x.sample(max(1, round(max_samples * len(x) / len(frame))), random_state=seed)).reset_index(drop=True)

def extract_embeddings(frame, cfg, kind, device, output_dir, split_name, model):
    cache_path = Path(output_dir) / f"pretrained_embedding_{kind}_{split_name}_{CACHE_VERSION}_{'full' if not MAX_SAMPLES else 'sample' + str(MAX_SAMPLES)}.npz"
    if cache_path.exists():
        cached = np.load(cache_path)
        return cached["x"], cached["y"]
    crop_mode = cfg.train_crop if split_name == "train" else cfg.eval_crop
    dataset = AudioVisualDataset(frame, cfg, crop_mode)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    xs, ys = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"extract {kind} {split_name}"):
            emb = model(batch["waveform"].to(device), batch["audio"].to(device), batch["image"].to(device), kind)
            xs.append(emb.cpu().numpy())
            ys.append(batch["label"].numpy())
    x = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    np.savez_compressed(cache_path, x=x, y=y)
    return x, y

def make_classifier(name):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import LinearSVC
    if name == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
    if name == "linear_svm":
        return make_pipeline(StandardScaler(), LinearSVC(class_weight="balanced", max_iter=5000))
    if name == "random_forest":
        return RandomForestClassifier(n_estimators=250, random_state=SEED, class_weight="balanced_subsample", n_jobs=-1)
    if name == "xgboost":
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, objective="multi:softmax", num_class=len(LABELS), eval_metric="mlogloss", random_state=SEED, n_jobs=-1)
    raise ValueError(name)

def compute_metrics(y_true, y_pred):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(LABELS))), average=PAPER_AVERAGE, zero_division=0)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)
    binary_true = (np.asarray(y_true) != LABEL_TO_ID["ambient"]).astype(int)
    binary_pred = (np.asarray(y_pred) != LABEL_TO_ID["ambient"]).astype(int)
    binary_precision, binary_recall, binary_f1, _ = precision_recall_fscore_support(binary_true, binary_pred, average="binary", zero_division=0)
    return {"paper_f1": float(f1), "paper_precision": float(precision), "paper_recall": float(recall), "macro_f1": float(macro_f1), "macro_precision": float(macro_precision), "macro_recall": float(macro_recall), "accuracy": float(accuracy_score(y_true, y_pred)), "binary_contact_f1": float(binary_f1), "binary_contact_precision": float(binary_precision), "binary_contact_recall": float(binary_recall)}

data_root = find_data_root(DATASET_ROOT)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = DataConfig(data_root=data_root)
index = build_index(cfg.data_root, cfg.skip_missing_files)
train_df = sample_stratified_frame(index[index.split_name == "audio_visual_dataset_default"], MAX_SAMPLES, SEED)
val_df = sample_stratified_frame(index[index.split_name == "audio_visual_dataset_robo_default"], MAX_SAMPLES, SEED + 1)
print("DATA_ROOT =", cfg.data_root)
print("OUTPUT_DIR =", output_dir)
print("device =", device)
print("train/val =", len(train_df), len(val_df))

extractor = PretrainedEmbeddingExtractor(cfg.target_sample_rate).to(device).eval()
rows = []
for kind in EMBEDDING_KINDS:
    x_train, y_train = extract_embeddings(train_df, cfg, kind, device, output_dir, "train", extractor)
    x_val, y_val = extract_embeddings(val_df, cfg, kind, device, output_dir, "val", extractor)
    clf = make_classifier(CLASSIFIER_NAME)
    clf.fit(x_train, y_train)
    pred = clf.predict(x_val)
    metrics = compute_metrics(y_val, pred)
    result = {"group": "pretrained_embedding_ml", "mode": f"pretrained_embedding_{kind}_{CLASSIFIER_NAME}", "input": f"{kind}_embedding", "feature": f"{kind}_embedding", "encoder": "AST+CLAP+ViT" if kind == "fusion" else "AST+CLAP" if kind == "audio" else "ViT", "fusion": kind if kind == "fusion" else "none", "classifier": CLASSIFIER_NAME, "pretrained": True, "frozen": True, "seed": SEED, "best_val_metrics": metrics}
    (output_dir / f"pretrained_embedding_{kind}_{CLASSIFIER_NAME}_results.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    row = {k: result[k] for k in ("group", "mode", "input", "feature", "encoder", "fusion", "classifier", "pretrained", "frozen", "seed")}
    row.update(metrics)
    rows.append(row)

results = pd.DataFrame(rows).sort_values("paper_f1", ascending=False).reset_index(drop=True)
results.to_csv(output_dir / f"pretrained_embedding_{CLASSIFIER_NAME}_results.csv", index=False)
results


DATA_ROOT = /kaggle/input/datasets/kurt54/visual-audio/img_audio
OUTPUT_DIR = /kaggle/working/outputs/variants
device = cuda
train/val = 10676 2218


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

extract video val:   0%|          | 0/555 [00:00<?, ?it/s]

extract fusion train:   0%|          | 0/2669 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7891458e8680>
Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7891458e8680>
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        if w.is_alive():if w.is_alive():

   Exception ignored in:    <function _MultiProcessingDataLoaderIter.__del__ at 0x7891458e8680>  
   Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", l

extract fusion val:   0%|          | 0/555 [00:00<?, ?it/s]

,group,mode,input,feature,encoder,fusion,classifier,pretrained,frozen,seed,paper_f1,paper_precision,paper_recall,macro_f1,macro_precision,macro_recall,accuracy,binary_contact_f1,binary_contact_precision,binary_contact_recall
0,pretrained_embedding_ml,pretrained_embedding_fusion_xgboost,fusion_embedding,fusion_embedding,AST+CLAP+ViT,fusion,xgboost,True,True,42,0.714921,0.736944,0.745266,0.634417,0.692588,0.631510,0.745266,0.852926,0.998765,0.744250
1,pretrained_embedding_ml,pretrained_embedding_audio_xgboost,audio_embedding,audio_embedding,AST+CLAP,none,xgboost,True,True,42,0.628321,0.626671,0.666817,0.484744,0.515181,0.498442,0.666817,0.878351,0.998828,0.783809
2,pretrained_embedding_ml,pretrained_embedding_video_xgboost,video_embedding,video_embedding,ViT,none,xgboost,True,True,42,0.351808,0.366414,0.512624,0.182264,0.327831,0.256384,0.512624,0.014585,0.800000,0.007360
